# Temporal Solar-Tornado Inference

This notebook runs the complete showcase pipeline: load a time-ordered AIA 171 Å sequence, extract or load the panoramic solar limb, divide the panorama into fixed spatial chunks, construct temporal difference images, detect tornadoes, and associate detections with ByteTrack IDs.

The checkpoint expects three channels: `current`, `abs(current - previous)`, and `abs(next - current)`. Keep the geometry and time-gap settings aligned with `training.ipynb`.

## 1. Setup

Install dependencies with `python -m pip install -r requirements.txt`, select this environment as the notebook kernel, and put input images under `data/raw/`.

In [ ]:
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import json
import re

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO

ROOT = Path.cwd().resolve()
if not (ROOT / 'models').exists() and (ROOT.parent / 'models').exists():
    ROOT = ROOT.parent

# Input can be full solar disks or precomputed panoramic boundary strips.
INPUT_DIR = ROOT / 'data' / 'raw'
INPUT_KIND = 'full_disk'  # 'full_disk' or 'limb_strip'
MODEL_PATH = ROOT / 'models' / 'temporal_model.pt'
LIMB_DIR = ROOT / 'artifacts' / 'inference' / 'limbs'
CHUNK_DIR = ROOT / 'artifacts' / 'inference' / 'chunks'
OUTPUT_PATH = ROOT / 'artifacts' / 'inference' / 'predictions.jsonl'

PANORAMA_WIDTH = 12_638
LIMB_HEIGHT = 384
OUTER_PIXELS = 320   # off-disk corona above the limb
INNER_PIXELS = 64    # bright disk retained below the limb
SEGMENT_COUNT = 20
CHUNK_WIDTH = 768
MAX_GAP_SECONDS = 120

CONFIDENCE = 0.25
NMS_IOU = 0.45
INFERENCE_BATCH = 8
TRACKER_WINDOW = 500  # bound memory while retaining tracker state
TRACKER = 'bytetrack.yaml'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
HALF = torch.cuda.is_available()

print(f'root: {ROOT}')
print(f'device: {DEVICE}')

## 2. Load the source sequence

A timestamp such as `2019-01-01T000012Z` must appear somewhere in every filename. Sorting by parsed timestamps—not lexicographic paths—prevents incorrect temporal neighbours.

In [ ]:
TIMESTAMP_RE = re.compile(r'(?P<ts>\d{4}-\d{2}-\d{2}T\d{6}Z)')

def timestamp_from_path(path: Path) -> tuple[str, datetime]:
    match = TIMESTAMP_RE.search(path.name)
    if match is None:
        raise ValueError(f'No YYYY-MM-DDTHHMMSSZ timestamp in {path.name}')
    text = match.group('ts')
    return text, datetime.strptime(text, '%Y-%m-%dT%H%M%SZ')

def read_grayscale_uint8(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if image is None:
        raise ValueError(f'OpenCV could not read {path}')
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    if image.dtype == np.uint8:
        return image
    finite = image[np.isfinite(image)]
    if finite.size == 0:
        raise ValueError(f'Image contains no finite pixels: {path}')
    lo, hi = np.percentile(finite, [0.5, 99.5])
    if hi <= lo:
        return np.zeros(image.shape, dtype=np.uint8)
    return np.clip((image.astype(np.float32) - lo) * 255.0 / (hi - lo), 0, 255).astype(np.uint8)

extensions = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
source_paths = [p for p in INPUT_DIR.rglob('*') if p.is_file() and p.suffix.lower() in extensions]
source_records = []
for path in source_paths:
    ts, dt = timestamp_from_path(path)
    source_records.append({'timestamp': ts, 'datetime': dt, 'path': path})
source_records.sort(key=lambda row: row['datetime'])

if not source_records:
    raise FileNotFoundError(f'No timestamped images found under {INPUT_DIR}')
if len({row['datetime'] for row in source_records}) != len(source_records):
    raise ValueError('Duplicate timestamps found; keep one source image per observation time.')

print(f'loaded {len(source_records):,} source frames')
print(f"range: {source_records[0]['timestamp']} → {source_records[-1]['timestamp']}")

## 3. Extract the panoramic limb

For full-disk input, the largest bright region estimates the disk center and radius. The remap then samples concentric rings: off-limb corona at the top, solar disk at the bottom. Inspect the result below. If automatic fitting is unreliable, pass an explicit `(center_x, center_y)` and radius. Precomputed `image_lev1_boundary` products can skip this stage with `INPUT_KIND = 'limb_strip'`.

In [ ]:
def estimate_solar_disk(image: np.ndarray) -> tuple[tuple[float, float], float]:
    blurred = cv2.GaussianBlur(image, (0, 0), sigmaX=5)
    _, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise ValueError('Could not estimate the solar disk; set center/radius explicitly.')
    contour = max(contours, key=cv2.contourArea)
    (center_x, center_y), radius = cv2.minEnclosingCircle(contour)
    if radius < 0.2 * min(image.shape):
        raise ValueError('Estimated disk is implausibly small; set center/radius explicitly.')
    return (float(center_x), float(center_y)), float(radius)

def extract_limb_panorama(
    full_disk: np.ndarray,
    center: tuple[float, float] | None = None,
    radius: float | None = None,
) -> tuple[np.ndarray, tuple[float, float], float]:
    if center is None or radius is None:
        center, radius = estimate_solar_disk(full_disk)
    angles = np.linspace(0.0, 2.0 * np.pi, PANORAMA_WIDTH, endpoint=False, dtype=np.float32)
    # Large-to-small radius makes off-limb structures appear above the bright disk.
    radii = np.linspace(radius + OUTER_PIXELS, radius - INNER_PIXELS, LIMB_HEIGHT, dtype=np.float32)
    map_x = center[0] + radii[:, None] * np.cos(angles)[None, :]
    map_y = center[1] + radii[:, None] * np.sin(angles)[None, :]
    panorama = cv2.remap(
        full_disk, map_x, map_y, interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0,
    )
    return panorama, center, radius

LIMB_DIR.mkdir(parents=True, exist_ok=True)
limb_records = []
for row in tqdm(source_records, desc='extracting limbs'):
    source = read_grayscale_uint8(row['path'])
    if INPUT_KIND == 'full_disk':
        limb, center, radius = extract_limb_panorama(source)
        limb_path = LIMB_DIR / f"{row['timestamp']}.png"
        if not cv2.imwrite(str(limb_path), limb):
            raise OSError(f'Failed to write {limb_path}')
    elif INPUT_KIND == 'limb_strip':
        limb = source
        center = radius = None
        limb_path = row['path']
        if limb.shape[0] != LIMB_HEIGHT:
            raise ValueError(f'{limb_path.name}: expected height {LIMB_HEIGHT}, got {limb.shape[0]}')
    else:
        raise ValueError("INPUT_KIND must be 'full_disk' or 'limb_strip'")
    if limb.shape[1] < PANORAMA_WIDTH:
        raise ValueError(f'{limb_path.name}: panorama is narrower than {PANORAMA_WIDTH}px')
    limb_records.append({**row, 'limb_path': limb_path, 'center': center, 'radius': radius})

sample = limb_records[len(limb_records) // 2]
sample_limb = read_grayscale_uint8(sample['limb_path'])
fig, ax = plt.subplots(figsize=(16, 3))
ax.imshow(sample_limb, cmap='gray', aspect='auto', vmin=0, vmax=255)
ax.set(title=f"Panoramic limb — {sample['timestamp']}", xlabel='angular position (px)', ylabel='radial position (px)')
plt.show()

## 4. Chunk each limb panorama

Twenty fixed angular positions are used in every timestamp. Adjacent 768-pixel chunks overlap because their starts are 631 pixels apart. The seam chunk wraps to the start of the panorama.

In [ ]:
def chunk_limb(limb: np.ndarray) -> list[np.ndarray]:
    starts = np.arange(SEGMENT_COUNT) * (PANORAMA_WIDTH // SEGMENT_COUNT)
    chunks = []
    for start in starts:
        indices = (np.arange(CHUNK_WIDTH) + int(start)) % limb.shape[1]
        chunks.append(limb[:, indices])
    return chunks

CHUNK_DIR.mkdir(parents=True, exist_ok=True)
chunk_records = []
for row in tqdm(limb_records, desc='chunking limbs'):
    limb = read_grayscale_uint8(row['limb_path'])
    for segment_id, chunk in enumerate(chunk_limb(limb)):
        chunk_path = CHUNK_DIR / f"{row['timestamp']}_{segment_id}.png"
        if not cv2.imwrite(str(chunk_path), chunk):
            raise OSError(f'Failed to write {chunk_path}')
        chunk_records.append({
            'timestamp': row['timestamp'], 'datetime': row['datetime'],
            'segment_id': segment_id, 'path': chunk_path,
        })

print(f'wrote {len(chunk_records):,} chunks to {CHUNK_DIR}')

## 5. Build temporal frames

Frames are grouped by angular segment and sorted by observation time. Neighbours across gaps larger than two minutes are deliberately ignored.

In [ ]:
segment_sequences = defaultdict(list)
for row in chunk_records:
    segment_sequences[row['segment_id']].append(row)
for sequence in segment_sequences.values():
    sequence.sort(key=lambda row: row['datetime'])

def temporal_frame(sequence: list[dict], index: int) -> np.ndarray:
    current = read_grayscale_uint8(sequence[index]['path'])
    previous = current
    following = current
    if index > 0:
        gap = (sequence[index]['datetime'] - sequence[index - 1]['datetime']).total_seconds()
        if gap <= MAX_GAP_SECONDS:
            previous = read_grayscale_uint8(sequence[index - 1]['path'])
    if index + 1 < len(sequence):
        gap = (sequence[index + 1]['datetime'] - sequence[index]['datetime']).total_seconds()
        if gap <= MAX_GAP_SECONDS:
            following = read_grayscale_uint8(sequence[index + 1]['path'])
    return cv2.merge([
        current, cv2.absdiff(current, previous), cv2.absdiff(following, current)
    ])

preview_sequence = segment_sequences[0]
preview_index = min(len(preview_sequence) // 2, len(preview_sequence) - 1)
preview_temporal = temporal_frame(preview_sequence, preview_index)
titles = ['current', '|current - previous|', '|next - current|']
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for channel, (ax, title) in enumerate(zip(axes, titles)):
    ax.imshow(preview_temporal[:, :, channel], cmap='gray', vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Detect and track

Tracker state is reset between angular segments because they represent different regions. It persists across bounded windows inside one segment. The output is JSONL for easy streaming into later analysis.

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(MODEL_PATH)

model = YOLO(str(MODEL_PATH))
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
prediction_records = []

with OUTPUT_PATH.open('w', encoding='utf-8') as output_file:
    for segment_id in tqdm(sorted(segment_sequences), desc='limb segments'):
        sequence = segment_sequences[segment_id]
        model.predictor = None  # do not carry tracks across spatially unrelated segments
        for start in range(0, len(sequence), TRACKER_WINDOW):
            stop = min(start + TRACKER_WINDOW, len(sequence))
            temporal_batch = [temporal_frame(sequence, index) for index in range(start, stop)]
            results = model.track(
                source=temporal_batch, stream=True, persist=True, tracker=TRACKER,
                imgsz=CHUNK_WIDTH, batch=INFERENCE_BATCH, conf=CONFIDENCE,
                iou=NMS_IOU, device=DEVICE, half=HALF, verbose=False, save=False,
            )
            for offset, result in enumerate(results):
                row = sequence[start + offset]
                boxes = result.boxes
                xyxy = boxes.xyxy.cpu().tolist() if boxes is not None else []
                confidence = boxes.conf.cpu().tolist() if boxes is not None and boxes.conf is not None else []
                classes = boxes.cls.cpu().tolist() if boxes is not None and boxes.cls is not None else []
                track_ids = (
                    boxes.id.int().cpu().tolist()
                    if boxes is not None and getattr(boxes, 'id', None) is not None
                    else [None] * len(xyxy)
                )
                record = {
                    'path': str(row['path']), 'timestamp': row['timestamp'],
                    'segment_id': segment_id, 'boxes_xyxy': xyxy,
                    'classes': classes, 'confidence': confidence, 'track_ids': track_ids,
                }
                prediction_records.append(record)
                output_file.write(json.dumps(record) + '\n')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

detection_count = sum(len(row['boxes_xyxy']) for row in prediction_records)
print(f'wrote {len(prediction_records):,} frame records / {detection_count:,} detections')
print(OUTPUT_PATH)

## 7. Inspect results

The example below selects the frame with the most detections and overlays confidence and track ID on the ordinary current-frame channel.

In [ ]:
if prediction_records:
    example = max(prediction_records, key=lambda row: len(row['boxes_xyxy']))
    image = read_grayscale_uint8(Path(example['path']))
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(image, cmap='gray', vmin=0, vmax=255)
    for box, confidence, track_id in zip(
        example['boxes_xyxy'], example['confidence'], example['track_ids']
    ):
        x1, y1, x2, y2 = box
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='#48ff70', linewidth=1.5
        ))
        label = f'{confidence:.2f}' if track_id is None else f'id {track_id} · {confidence:.2f}'
        ax.text(x1, max(0, y1 - 4), label, color='#48ff70', fontsize=8,
                bbox={'facecolor': 'black', 'alpha': 0.55, 'pad': 1, 'edgecolor': 'none'})
    ax.set_title(f"{example['timestamp']} · segment {example['segment_id']}")
    ax.axis('off')
    plt.show()
else:
    print('No frame records were produced.')